# Step 5 (Phase 3) — LoRA + BitFit + Full-FT + Linear-Probe on CIFAR-FS

Git-based runner. Clones the repo into `/content` (fast local disk — avoids the
Drive-FUSE per-file hang from `instructions.txt` gotcha (b)), builds the frozen
Bertinetto split, and runs all **8** configs (4 adapters × 2 heads) end-to-end,
producing `results/phase3_*_metrics.json`.

**PREREQUISITE:** the Step 5 code must be pushed to the branch you set in
`BRANCH` below (default `main`). By repo convention a human commits/pushes —
do that first, then run this notebook top-to-bottom.

Runtime: LoRA / BitFit / Linear-Probe are close to Bottleneck's cost; the two
**Full-FT** runs are the slow ones (backprop through the whole backbone) and may
take ~20-40 min each on a T4. The run loop continues on error, so one failed
config never blocks the others.

## 0. GPU + environment check

In [ ]:
import torch, sys
print('python :', sys.version.split()[0])
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > T4 GPU'

## 1. Clone the repo + install deps

Set `BRANCH` to whatever branch has the Step 5 commit. Re-running the cell
pulls the latest instead of re-cloning.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/notAvailable73/thesis.git'
BRANCH   = 'main'          # <-- set to the branch that has your Step 5 push
REPO_DIR = '/content/thesis'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())
!pip -q install -r requirements.txt

# Sanity: the Step 5 adapters must be present in this checkout.
import importlib.util
for f in ['src/adapters/lora.py', 'src/adapters/bitfit.py', 'src/adapters/full_ft.py',
          'src/adapters/linear_probe.py']:
    assert os.path.exists(f), f'MISSING {f} — did you push the Step 5 commit to {BRANCH}?'
print('Step 5 adapters present — good.')

## 2. Build the frozen CIFAR-FS Bertinetto split

`data/` is gitignored, so materialize the canonical 64/16/20 split once
(downloads CIFAR-100 to local disk). Do NOT hand-edit the JSON it writes.

In [ ]:
!python scripts/build_cifar_fs_split.py
import json
sp = json.load(open('data/cifar_fs_split.json'))
print('split status:', sp.get('_status'))
print('sizes:', {k: len(v) for k, v in sp.items() if isinstance(v, list)})

## 3. (optional) Run the test suite

Confirms the new adapters + the rest of the suite pass before burning GPU on
the runs. Expect the Step-4/4.5 tests plus the four new Step-5 files.

In [ ]:
# Pre-fetch the frozen backbone's ImageNet weights (~45 MB) with a VISIBLE
# progress bar. The test suite builds the real ResNet-18, which otherwise
# downloads these weights silently on first use — on Colab's slow link to
# download.pytorch.org that looks exactly like a hang.
from torchvision.models import resnet18, ResNet18_Weights
_ = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
print('resnet18 ImageNet weights cached.')

# Run pytest UNBUFFERED and streamed (no `| tail`, which hides all output
# until the run finishes). `-v` prints one line per test so you can see it
# is alive and which test — if any — is actually slow.
!python -u -m pytest -v --durations=10

## 4. Run all 8 configs (train + 600-episode eval)

One JSON per config in `results/phase3_*_metrics.json`. Already-finished
configs are skipped, so you can re-run this cell to resume after an
interruption. `USE_TINYIMAGENET=True` adds the near-OOD pool (downloads a
~240 MB zip once); set it to `False` if that download is flaky.

In [ ]:
import subprocess, os

USE_TINYIMAGENET = True
NUM_EPISODES     = 600
SUFFIX           = 'phase3'

# (config-basename, adapter.type, interpretation)
RUNS = [
    ('exp_phase3_lora_evidential',        'lora',         'evidential'),
    ('exp_phase3_lora_softmax',           'lora',         'softmax'),
    ('exp_phase3_bitfit_evidential',      'bitfit',       'evidential'),
    ('exp_phase3_bitfit_softmax',         'bitfit',       'softmax'),
    ('exp_phase3_full_ft_evidential',     'full_ft',      'evidential'),
    ('exp_phase3_full_ft_softmax',        'full_ft',      'softmax'),
    ('exp_phase3_linear_probe_evidential','linear_probe', 'evidential'),
    ('exp_phase3_linear_probe_softmax',   'linear_probe', 'softmax'),
]

def result_path(adapter, interp):
    return f'results/{SUFFIX}_{adapter}_prototype-{interp}_metrics.json'

def run(cmd):
    # Inherit stdout/stderr so the training log streams live into the cell
    # (do NOT capture — captured subprocess output can silently vanish on
    # Colab; see instructions.txt gotcha (d)).
    print('>>>', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

status = {}
for name, adapter, interp in RUNS:
    out = result_path(adapter, interp)
    if os.path.exists(out):
        print(f'== SKIP {name} (found {out}) ==', flush=True)
        status[name] = 'skip (exists)'
        continue
    print(f'\n{"="*72}\n== {name} ==\n{"="*72}', flush=True)
    cfg = f'configs/{name}.yaml'
    try:
        rc = run(['python', 'scripts/train.py', '--config', cfg, '--wandb-mode', 'disabled'])
        if rc != 0:
            status[name] = f'TRAIN failed (rc={rc})'; continue
        eval_cmd = ['python', 'scripts/evaluate.py', '--config', cfg,
                    '--num-episodes', str(NUM_EPISODES), '--wandb-mode', 'disabled',
                    '--results-suffix', SUFFIX]
        if USE_TINYIMAGENET:
            eval_cmd.append('--use-tinyimagenet')
        rc = run(eval_cmd)
        status[name] = 'OK' if (rc == 0 and os.path.exists(out)) else f'EVAL failed (rc={rc})'
    except Exception as e:
        status[name] = f'EXCEPTION: {e}'

print('\n' + '=' * 40 + '\nRUN STATUS\n' + '=' * 40)
for name, _, _ in RUNS:
    print(f'  {name:38s} {status.get(name, "not run")}')

## 5. Summary table

Reads every `results/phase3_*_metrics.json` and shows the headline numbers,
alongside the Step 4.5 Bottleneck baseline of record.

In [ ]:
import glob, json
import pandas as pd

rows = []
for f in sorted(glob.glob('results/phase3_*_metrics.json')) + \
         sorted(glob.glob('results/step45_*_metrics.json')):
    d = json.load(open(f))
    rows.append({
        'file'      : os.path.basename(f),
        'adapter'   : d.get('adapter_type'),
        'interp'    : d.get('interpretation'),
        'n_params'  : d.get('n_params'),
        'accuracy'  : round(d.get('accuracy_mean', float('nan')), 4),
        'f1_macro'  : round(d.get('f1_macro_mean', float('nan')), 4),
        'ece'       : round(d.get('ece_pooled', float('nan')), 4),
        'ece_ts'    : round(d['ece_ts'], 4) if 'ece_ts' in d else None,
        'brier'     : round(d.get('brier_mean', float('nan')), 4),
        'auroc(prim)': round(d.get('ood_auroc_mean', float('nan')), 4),
        'best_val_ep': d.get('best_val_epoch'),
    })
df = pd.DataFrame(rows)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
df

## 6. (optional) Save results back to Drive

Colab's `/content` is wiped when the runtime ends. Run this to copy the 8
JSONs (+ PNGs) to your Drive so they survive; or download the zip.

In [ ]:
SAVE_TO_DRIVE = True
DRIVE_DEST = '/content/drive/MyDrive/bpeft_step5_results'

import shutil, glob, os
files = glob.glob('results/phase3_*')
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DEST, exist_ok=True)
    for f in files:
        shutil.copy(f, DRIVE_DEST)
    print(f'copied {len(files)} files to {DRIVE_DEST}')
else:
    shutil.make_archive('/content/phase3_results', 'zip', 'results')
    from google.colab import files as colab_files
    colab_files.download('/content/phase3_results.zip')